# LDPC Code Rate Adaptation — Full Baseline (Jupyter)

Полноценный ноутбук для подбора параметров `{R, s, p}` по данным квантового канала.

**Что внутри:**
1. Обзор задачи и метрики
2. Установка окружения (если нужно)
3. Загрузка данных (ваш `data.csv` или демо)
4. Признаки: лаги/скользящие по QBER и физпараметрам
5. Прогноз QBER (квантили 0.5 и 0.9) — GradientBoosting (quantile loss)
6. Суррогатная модель вероятности фейла коррекции (если есть `synErr`/`N_EC_rounds`)
7. Политика выбора `{R, s, p}` с fail-safe
8. Визуализация и сохранение результатов (`ldpc_predictions.csv`)

Запускайте ячейки сверху вниз. Если `data.csv` не найден, генерируется синтетика для демонстрации.

## 1) Импорт и настройки

In [ ]:
import os, math, json
import numpy as np
import pandas as pd
from dataclasses import dataclass
from typing import List, Tuple, Optional

from sklearn.ensemble import GradientBoostingRegressor
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

import matplotlib.pyplot as plt  # графики (без seaborn)

R_VALUES = [0.5, 0.55, 0.6, 0.65, 0.7, 0.75, 0.8, 0.85, 0.9]
plt.rcParams['figure.figsize'] = (8,4)

## 2) Утилиты: безопасные колонки, лаги, скользящие

In [ ]:
def safe_cols(df: pd.DataFrame, cols: List[str]) -> List[str]:
    return [c for c in cols if c in df.columns]

def add_lags(df: pd.DataFrame, col: str, lags: List[int]) -> pd.DataFrame:
    for L in lags:
        df[f"{col}_lag{L}"] = df[col].shift(L)
    return df

def add_rolls(df: pd.DataFrame, col: str, windows: List[int]) -> pd.DataFrame:
    for w in windows:
        df[f"{col}_roll{w}_mean"] = df[col].rolling(w, min_periods=1).mean()
        df[f"{col}_roll{w}_std"]  = df[col].rolling(w, min_periods=1).std().fillna(0.0)
    return df

def nearest_R(x: float) -> float:
    return min(R_VALUES, key=lambda r: abs(r - x))

## 3) Построение признаков и целевой для QBER

In [ ]:
def build_features(df: pd.DataFrame):
    # --- нормализуем имена (скобки -> _) ---
    df.columns = (
        df.columns
        .str.replace("[", "_", regex=False)
        .str.replace("]", "_", regex=False)
        .str.replace(" ", "_", regex=False)
        .str.replace(":", "_", regex=False)
        .str.replace(",", "_", regex=False)
    )

    # --- целевая: берём E_mu_Z_est ТОЛЬКО если она «здоровая», иначе E_mu_Z ---
    use_est = False
    if "E_mu_Z_est" in df.columns:
        y_est = pd.to_numeric(df["E_mu_Z_est"], errors="coerce")
        if y_est.notna().sum() > 100 and y_est.median() < 0.18 and y_est.std() > 1e-4:
            use_est = True

    if use_est:
        target_col = "E_mu_Z_est"
        if "E_mu_Z" in df.columns:
            m = df[target_col].isna()
            df.loc[m, target_col] = df.loc[m, "E_mu_Z"]
    elif "E_mu_Z" in df.columns:
        target_col = "E_mu_Z"
    else:
        target_col = "E_mu_Z"
        df[target_col] = np.nan

    # --- упорядочим по времени (если есть) ---
    time_cols = [c for c in ["block_id","frame_idx"] if c in df.columns]
    if time_cols:
        df = df.sort_values(time_cols).reset_index(drop=True)

    # --- БАЗОВЫЕ ПРИЗНАКИ (с учётом нормализованных имён) ---
    base_cols = safe_cols(df, [
        "E_mu_Z","E_mu_phys_est","E_mu_X","E_nu1_X","E_nu2_X","E_nu1_Z","E_nu2_Z",
        "N_mu_X","M_mu_XX","M_mu_XZ","M_mu_X","N_mu_Z","M_mu_ZZ","M_mu_Z",
        "N_nu1_X","M_nu1_XX","M_nu1_XZ","M_nu1_X","N_nu1_Z","M_nu1_ZZ","M_nu1_Z",
        "N_nu2_X","M_nu2_XX","M_nu2_XZ","M_nu2_X","N_nu2_Z","M_nu2_ZZ","M_nu2_Z",
        "nTot","unitsRatio","opticalPower","bayesImVoltage",
        "polarizerVoltages_0_","polarizerVoltages_1_","polarizerVoltages_2_","polarizerVoltages_3_",
        "temp_1","biasVoltage_1","temp_2","biasVoltage_2",
        "synErr","N_EC_rounds","f_EC"
    ])

    # --- приведение типов к числам ДО инженерии ---
    num_all = [c for c in df.columns if c != "estimator_name"]
    df[num_all] = df[num_all].apply(pd.to_numeric, errors="coerce")

    # --- инженерия вокруг таргета ---
    df = add_lags(df, target_col, [1,2,3,5,10])
    df = add_rolls(df, target_col, [3,5,9,15])
    eng_cols = [c for c in df.columns if c.startswith(f"{target_col}_lag") or c.startswith(f"{target_col}_roll")]

    feature_cols = list(dict.fromkeys(base_cols + eng_cols))

    # --- сборка матриц (и без deprecated fillna(method='ffill')) ---
    X = df[feature_cols].ffill().fillna(0.0)
    y = pd.to_numeric(df[target_col], errors="coerce").ffill()
    if y.isna().any():
        y = y.fillna(float(np.nanmedian(y)))

    # ограничим y разумным диапазоном [0, 0.2] (QBER)
    y = np.clip(y, 0.0, 0.2)

    return X, y, feature_cols


In [ ]:
TARGET_START = (1489460492, 99)   # (block_id, frame_idx)
TARGET_END   = (1840064900, 101)

def slice_target_2000(df: pd.DataFrame) -> pd.DataFrame:
    # Никаких сортировок! Сохраняем исходный порядок файла.
    df = df.reset_index(drop=True)
    start_idx = df.index[(df["block_id"] == TARGET_START[0]) & (df["frame_idx"] == TARGET_START[1])]
    end_idx   = df.index[(df["block_id"] == TARGET_END[0])   & (df["frame_idx"] == TARGET_END[1])]
    if len(start_idx)==0 or len(end_idx)==0:
        raise ValueError("Не нашли старт/финиш в файле. Проверь, тот ли CSV.")
    start_idx, end_idx = int(start_idx[0]), int(end_idx[0])
    if end_idx < start_idx:
        raise ValueError("END раньше START — проверь файл/порядок.")
    win = df.iloc[start_idx:end_idx+1].copy()
    assert len(win) == 2000, f"Ожидали 2000 строк, получили {len(win)}"
    return win

In [ ]:
from sklearn.model_selection import TimeSeriesSplit
import numpy as np

def tscv_splits(X, n_splits=5, test_size=400, gap=2):
    tscv = TimeSeriesSplit(n_splits=n_splits, test_size=test_size, gap=gap)
    for tr, va in tscv.split(X):
        yield tr, va

# === Optuna для LightGBM-квантилей ===
import optuna
from lightgbm import LGBMRegressor
from sklearn.metrics import mean_absolute_error

def tune_lgbm_quantile(X, y, alpha=0.8, n_trials=60):
    def objective(trial):
        params = {
            "objective": "quantile",
            "alpha": alpha,
            "num_leaves": trial.suggest_int("num_leaves", 15, 255),
            "min_data_in_leaf": trial.suggest_int("min_data_in_leaf", 20, 300),
            "feature_fraction": trial.suggest_float("feature_fraction", 0.6, 1.0),
            "bagging_fraction": trial.suggest_float("bagging_fraction", 0.6, 1.0),
            "lambda_l1": trial.suggest_float("lambda_l1", 0.0, 2.0),
            "lambda_l2": trial.suggest_float("lambda_l2", 0.0, 2.0),
            "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
            "n_estimators": trial.suggest_int("n_estimators", 300, 1800),
            "random_state": 42,
            "verbosity": -1,
        }
        maes = []
        for tr, va in tscv_splits(X):
            model = LGBMRegressor(**params)
            model.fit(X.iloc[tr], y.iloc[tr])
            pred = model.predict(X.iloc[va])
            maes.append(mean_absolute_error(y.iloc[va], pred))
        return float(np.mean(maes))
    study = optuna.create_study(direction="minimize")
    study.optimize(lambda t: objective(t), n_trials=n_trials, show_progress_bar=False)
    best = study.best_params
    best.update({"objective":"quantile","alpha":alpha,"random_state":42,"verbosity":-1})
    return best

# Тренируем две головы: медиана (0.5) и «верхний» квантиль (0.8 или 0.9)
def fit_quantile_heads(X, y, alpha_hi=0.8):
    best50 = tune_lgbm_quantile(X, y, alpha=0.5, n_trials=40)
    bestHI = tune_lgbm_quantile(X, y, alpha=alpha_hi, n_trials=40)
    m50 = LGBMRegressor(**best50).fit(X, y)
    mHI = LGBMRegressor(**bestHI).fit(X, y)
    return m50, mHI

## 4) Модель прогноза QBER (квантили 0.5 и 0.9)

In [ ]:
@dataclass
class QBERForecaster:
    q50: Optional[GradientBoostingRegressor] = None
    q90: Optional[GradientBoostingRegressor] = None
    feature_cols: Optional[List[str]] = None

    def fit(self, X: pd.DataFrame, y: pd.Series):
        self.feature_cols = list(X.columns)
        self.q50 = GradientBoostingRegressor(loss='quantile', alpha=0.5, random_state=42, n_estimators=300, max_depth=3)
        self.q90 = GradientBoostingRegressor(loss='quantile', alpha=0.9, random_state=42, n_estimators=300, max_depth=3)
        self.q50.fit(X, y)
        self.q90.fit(X, y)

    def predict_quantiles(self, X: pd.DataFrame) -> Tuple[np.ndarray, np.ndarray]:
        X_ = X[self.feature_cols]
        q_med = self.q50.predict(X_)
        q_hi  = self.q90.predict(X_)
        q_med = np.clip(q_med, 0.0, 0.2)
        q_hi  = np.clip(q_hi,  0.0, 0.2)
        return q_med, q_hi

## 5) Суррогатная модель вероятности фейла (опционально)

In [ ]:
@dataclass
class FailSurrogate:
    model: Optional[Pipeline] = None
    feature_cols: Optional[List[str]] = None

    def fit(self, df: pd.DataFrame, base_features: List[str]) -> bool:
        labels_available = ('synErr' in df.columns) or ('N_EC_rounds' in df.columns)
        params_available = all(c in df.columns for c in ['R', 's'])
        if not labels_available or not params_available:
            return False

        if 'synErr' in df.columns:
            y = (df['synErr'] > 0).astype(int)
        else:
            y = (df['N_EC_rounds'] > df['N_EC_rounds'].median()).astype(int)

        feats = base_features + ['R','s']
        feats = safe_cols(df, feats)
        clean = df.dropna(subset=feats + [y.name])
        if clean.empty:
            return False

        X = clean[feats]
        y = clean[y.name]

        self.model = Pipeline([
            ('scaler', StandardScaler(with_mean=False)),
            ('clf', LogisticRegression(max_iter=200))
        ])
        self.model.fit(X, y)
        self.feature_cols = feats
        return True

    def predict_fail_prob(self, X_row: pd.Series, R: float, s: int, q_hi: float) -> float:
        if self.model is None or self.feature_cols is None:
            base_risk = np.clip((q_hi - 0.02) * 20.0, 0.0, 1.0)
            r_term = np.clip((R - 0.6) * 2.0, -1.0, 1.0)
            s_term = np.clip((2000 - s) / 2000.0, -1.0, 1.0)
            return float(np.clip(base_risk + 0.3*r_term + 0.3*s_term, 0.0, 1.0))

        x = X_row.copy()
        for c in self.feature_cols:
            if c not in x.index:
                x[c] = 0.0
        x = x[self.feature_cols].copy()
        if 'R' in x.index:
            x['R'] = R
        if 's' in x.index:
            x['s'] = s
        prob = self.model.predict_proba([x.values])[0][1]
        return float(prob)

## 6) Политика выбора `{R, s, p}`

In [ ]:
# @dataclass
# class LDPCPolicy:
#     lam: float = 0.4
#     mu: float = 3.0
#     base_alpha0: int = 800
#     base_alpha1: int = 300
#     beta_R: float = 0.6
#     s_min: int = 200
#     s_max: int = 4200
#     fail_threshold: float = 0.15

#     def allowed_R_by_q(self, q_hi: float) -> List[float]:
#         if q_hi < 0.015: return [0.85, 0.9]
#         if q_hi < 0.025: return [0.75, 0.8, 0.85]
#         if q_hi < 0.035: return [0.65, 0.7, 0.75]
#         if q_hi < 0.05:  return [0.55, 0.6, 0.65]
#         return [0.5]

#     def compute_s_base(self, q_hi: float, R: float) -> int:
#         s_base = self.base_alpha0 + self.base_alpha1 * (q_hi * 100.0)
#         s_corr = s_base - self.beta_R * (0.75 - R) * 1000.0
#         return int(np.clip(round(s_corr), self.s_min, self.s_max))

#     def neighbor_grid(self, s0: int) -> List[int]:
#         steps = [-400, -200, 0, 200, 400]
#         return sorted(set(int(np.clip(s0 + d, 0, 4800)) for d in steps))

#     def select(self, X_row: pd.Series, q_hi: float, surrogate: 'FailSurrogate') -> Tuple[float,int,int]:
#         best = None
#         for R in self.allowed_R_by_q(q_hi):
#             s0 = self.compute_s_base(q_hi, R)
#             for s in self.neighbor_grid(s0):
#                 p = 4800 - s
#                 fail_prob = surrogate.predict_fail_prob(X_row, R, s, q_hi)
#                 iters_hat = 30 + 150*q_hi + 0.0005*p - 20*(0.65 - R)
#                 skl_hat = (27200.0 * R) * (1.0 - 0.6*q_hi) * (1.0 - fail_prob)
#                 score = skl_hat - self.lam*iters_hat - self.mu*fail_prob*1000.0
#                 cand = (score, R, s, p, fail_prob)
#                 if (best is None) or (score > best[0]):
#                     best = cand
#         score, R, s, p, fail_prob = best
#         if fail_prob > self.fail_threshold:
#             idx = R_VALUES.index(nearest_R(R))
#             if idx > 0:
#                 R = R_VALUES[idx-1]
#             s = min(4800, s + 400)
#             p = 4800 - s
#         return float(R), int(s), int(p)

In [ ]:
@dataclass
class LDPCPolicy:
    fec: float = 1.12        # подберите на CV (1.10–1.16 разумно)
    delta: float = 0.02      # запас против фейлов
    s_grid: int = 100        # шаг подбора s
    fail_threshold: float = 0.15

    def H2(self, q: float) -> float:
        import math
        q = min(max(q, 1e-6), 1-1e-6)
        return -q*math.log2(q) - (1-q)*math.log2(1-q)

    def feasible_rates(self, q_hi: float):
        leak = self.fec * self.H2(q_hi)
        target_max_rate = 1.0 - leak - self.delta
        rates = [0.50,0.55,0.60,0.65,0.70,0.75,0.80,0.85,0.90]
        feas = [r for r in rates if r <= target_max_rate]
        return feas or [0.50]

    def choose(self, X_row: pd.Series, q_hi: float, surrogate: 'FailSurrogate'):
        # сетки
        R_cands = self.feasible_rates(q_hi)
        if   q_hi > 0.12: base_s = 2400
        elif q_hi > 0.08: base_s = 1800
        elif q_hi > 0.05: base_s = 1200
        else:             base_s = 800
        S_cands = list(range(max(0,base_s-400), min(4800,base_s+401), self.s_grid))

        best = None
        for R in R_cands:
            for s in S_cands:
                p = 4800 - s
                #fail_prob = surrogate.predict_fail_prob(X_row, R, s, q_hi)
                fail_prob = 0.0
                if surrogate and hasattr(surrogate, "predict_fail_prob"):
                    fail_prob = surrogate.predict_fail_prob(X_row, R, s, q_hi)
                # простой прокси-скор: меньше итераций/фейлов, больше секретный бит
                iters_hat = 30 + 150*q_hi + 0.0005*p - 20*(0.65 - R)
                skl_hat = (27200.0 * R) * (1.0 - 0.6*q_hi) * (1.0 - fail_prob)
                score = skl_hat - 0.4*iters_hat - 3.0*fail_prob*1000.0
                cand = (score, R, s, p, fail_prob)
                if (best is None) or (score > best[0]):
                    best = cand

        score, R, s, p, fail_prob = best
        if fail_prob > self.fail_threshold:
            # fallback — понижаем rate и поднимаем s
            rates = [0.50,0.55,0.60,0.65,0.70,0.75,0.80,0.85,0.90]
            idx = rates.index(R)
            if idx > 0:
                R = rates[idx-1]
            s = min(4800, s + 400)
            p = 4800 - s
        return float(R), int(s), int(p)

## 7) End-to-end: обучение, предсказание, сохранение

In [ ]:
# def train_qber_models(X: pd.DataFrame, y: pd.Series) -> QBERForecaster:
#     forecaster = QBERForecaster()
#     forecaster.fit(X, y)
#     return forecaster

def train_qber_models(X: pd.DataFrame, y: pd.Series) -> QBERForecaster:
    # Используем LGBM-квантили вместо sklearn GBDT
    m50, mHI = fit_quantile_heads(X, y, alpha_hi=0.8)
    # Оборачиваем в ваш QBERForecaster, чтобы интерфейс не менять
    f = QBERForecaster()
    f.feature_cols = list(X.columns)
    f.q50 = m50   # легально: атрибуты класса — просто держим модели
    f.q90 = mHI
    return f

def train_fail_surrogate(df: pd.DataFrame, feature_cols: List[str]) -> 'FailSurrogate':
    surrogate = FailSurrogate()
    surrogate.fit(df.copy(), feature_cols)
    return surrogate

# def run_policy_on_df(df: pd.DataFrame, forecaster: QBERForecaster, surrogate: 'FailSurrogate') -> pd.DataFrame:
#     X_all, _, _ = build_features(df.copy())
#     q_med, q_hi = forecaster.predict_quantiles(X_all)
#     policy = LDPCPolicy()
#     R_list, s_list, p_list = [], [], []
#     for i in range(len(df)):
#         row = X_all.iloc[i]
#         R, s, p = policy.select(row, q_hi=q_hi[i], surrogate=surrogate)
#         R_list.append(R); s_list.append(s); p_list.append(p)
#     out = df.copy()
#     out['QBER_med_pred'] = q_med
#     out['QBER_hi_pred']  = q_hi
#     out['R_pred'] = R_list
#     out['s_pred'] = s_list
#     out['p_pred'] = p_list
#     return out


def run_policy_on_df(df: pd.DataFrame,
                     forecaster: QBERForecaster,
                     surrogate: 'FailSurrogate',
                     rows_mask: Optional[pd.Series] = None) -> pd.DataFrame:
    """
    Если rows_mask передан, считаем предикты и политику только на подмножестве строк.
    """
    # 1) Фичи для ВСЕГО df (лагам нужна история)
    X_all, _, _ = build_features(df.copy())

    # 2) Выбираем подмножество, которое нужно скорить
    if rows_mask is None:
        rows_mask = pd.Series(True, index=df.index)
    X_sc = X_all[rows_mask]
    df_sc = df[rows_mask]

    # 3) Прогнозы квантилей на подмножестве (модель обучена на полном X/y)
    q_med, q_hi = forecaster.predict_quantiles(X_sc)

    # (опционально) EMA-сглаживание
    q_med = pd.Series(q_med, index=df_sc.index).ewm(alpha=0.3).mean().values
    q_hi  = pd.Series(q_hi,  index=df_sc.index).ewm(alpha=0.3).mean().values

    # 4) Политика — только по выбранным строкам
    policy = LDPCPolicy()
    R_list, s_list, p_list = [], [], []
    for i, (idx, X_row) in enumerate(X_sc.iterrows()):
        R, s, p = policy.choose(X_row, q_hi=float(q_hi[i]), surrogate=surrogate)
        R_list.append(R); s_list.append(s); p_list.append(p)

    # 5) Возвращаем ПРЕДИКТЫ ТОЛЬКО НА ПОДМНОЖЕСТВЕ
    out = df_sc.copy()
    out["QBER_med_pred"] = q_med
    out["QBER_hi_pred"]  = q_hi
    out["R_pred"]        = R_list
    out["s_pred"]        = s_list
    out["p_pred"]        = p_list
    return out



# def run_pipeline(input_csv: str = 'datasets/frames_errors.csv', output_csv: str = 'ldpc_predictions.csv') -> pd.DataFrame:
#     if os.path.exists(input_csv):
#         df = pd.read_csv(input_csv)
#         print(f"Read: {input_csv}, shape={df.shape}")
#     else:
#         # демо-данные
#         n = 400
#         rng = np.random.default_rng(7)
#         df = pd.DataFrame({
#             'block_id': np.repeat(np.arange(n//20), 20),
#             'frame_idx': np.arange(n),
#             'E_mu_Z': np.clip(rng.normal(0.025, 0.006, n), 0.0, 0.12),
#             'opticalPower': rng.normal(1.0, 0.05, n),
#             'temp_1': rng.normal(-50, 0.3, n),
#             'temp_2': rng.normal(-50, 0.3, n),
#             'bayesImVoltage': rng.normal(0.0, 0.1, n),
#             'polarizerVoltages[0]': rng.normal(0.0, 0.2, n),
#             'polarizerVoltages[1]': rng.normal(0.0, 0.2, n),
#             'polarizerVoltages[2]': rng.normal(0.0, 0.2, n),
#             'polarizerVoltages[3]': rng.normal(0.0, 0.2, n),
#             'synErr': rng.integers(0, 2, n),
#             'N_EC_rounds': rng.integers(1, 4, n),
#             'R': rng.choice(R_VALUES, size=n),
#             's': rng.integers(400, 3200, size=n)
#         })
#         print("No data.csv found — generated synthetic demo.")

#     X, y, feature_cols = build_features(df.copy())
#     forecaster = train_qber_models(X, y)
#     surrogate  = train_fail_surrogate(df.copy(), feature_cols)
#     pred_df = run_policy_on_df(df.copy(), forecaster, surrogate)
#     pred_df.to_csv(output_csv, index=False)
#     print(f"Saved predictions to: {output_csv}")
#     return pred_df

def run_pipeline(input_csv: str, output_csv: str):
    # --- 1) robust-чтение CSV: корректные имена + хедер/без хедера ---
    BASE_COLS = [
        "block_id","frame_idx",
        "E_mu_Z","E_mu_phys_est","E_mu_X","E_nu1_X","E_nu2_X","E_nu1_Z","E_nu2_Z",
        "N_mu_X","M_mu_XX","M_mu_XZ","M_mu_X","N_mu_Z","M_mu_ZZ","M_mu_Z",
        "N_nu1_X","M_nu1_XX","M_nu1_XZ","M_nu1_X","N_nu1_Z","M_nu1_ZZ","M_nu1_Z",
        "N_nu2_X","M_nu2_XX","M_nu2_XZ","M_nu2_X","N_nu2_Z","M_nu2_ZZ","M_nu2_Z",
        "nTot","unitsRatio","bayesImVoltage","opticalPower",
        "polarizerVoltages[0]","polarizerVoltages[1]","polarizerVoltages[2]","polarizerVoltages[3]",
        "temp_1","biasVoltage_1","temp_2","biasVoltage_2",
        "synErr","N_EC_rounds","estimator_name","f_EC","E_mu_Z_est",
        "R","s","p"
    ]
    def _read_csv_robust(path: str) -> pd.DataFrame:
        # пробуем как без заголовка
        df = pd.read_csv(path, header=None)
        if df.shape[1] == 50:
            df.columns = BASE_COLS
            # если первая «строка» выглядит как заголовок — перечитаем с header=0
            if isinstance(df.iloc[0,0], str) and str(df.iloc[0,0]).lower() == "block_id":
                df = pd.read_csv(path)
        else:
            df = pd.read_csv(path)  # запасной вариант
        return df

    df = _read_csv_robust(input_csv)

    # --- 2) типы: всё числовое, кроме estimator_name ---
    num_cols = [c for c in df.columns if c != "estimator_name"]
    df[num_cols] = df[num_cols].apply(pd.to_numeric, errors="coerce")
    if "block_id" in df:  df["block_id"]  = df["block_id"].astype("int64")
    if "frame_idx" in df: df["frame_idx"] = df["frame_idx"].astype("int64")

    # --- 3) тренировка предиктора QBER ---
    X_all, y_all, feature_cols = build_features(df.copy())
    forecaster = train_qber_models(X_all, y_all)

    # --- 4) суррогат фейлов (если метки есть) ---
    if "synErr" in df.columns:
        y_fail = (df["synErr"].fillna(0).astype(float) > 0).astype(int)
        surrogate = FailSurrogate().fit(X_all[feature_cols], y_fail)
    else:
        surrogate = FailSurrogate()

    # --- 5) скорим ровно нужные 2000 строк (по точным индексам окна) ---
    start_bid, start_fr = 1489460492, 99
    end_bid,   end_fr   = 1840064900, 101
    i0 = df.index[(df["block_id"]==start_bid) & (df["frame_idx"]==start_fr)]
    i1 = df.index[(df["block_id"]==end_bid)   & (df["frame_idx"]==end_fr)]
    if len(i0) and len(i1) and int(i1[0]) >= int(i0[0]):
        mask = df.index.isin(range(int(i0[0]), int(i1[0])+1))
    else:
        # фолбэк по диапазону, если вдруг индексы не нашлись
        mask = df["block_id"].between(start_bid, end_bid) & df["frame_idx"].between(start_fr, end_fr)

    pred_df = run_policy_on_df(df.copy(), forecaster, surrogate, rows_mask=mask)

    # --- 6) сохранить ---
    pred_df.to_csv(output_csv, index=False)
    print(f"Saved predictions to: {output_csv}")
    return pred_df




## 8) Визуализация: зависимость выбранного R от QBER (наглядно)

In [ ]:
def plot_R_vs_QBER(pred_df: pd.DataFrame):
    if not {'frame_idx','QBER_hi_pred','R_pred'}.issubset(pred_df.columns):
        print('Nothing to plot.')
        return
    fig = plt.figure()
    plt.scatter(pred_df['QBER_hi_pred'], pred_df['R_pred'], s=10)
    plt.xlabel('QBER (q90)')
    plt.ylabel('Chosen R')
    plt.title('Policy: chosen R vs predicted QBER (q90)')
    plt.grid(True)
    plt.show()

## 9) Запуск (демо)

In [ ]:
print("len(BASE_COLS):", len(BASE_COLS))

In [ ]:
import pandas as pd, time

def quick_test_pipeline(input_csv: str = "datasets/frames_errors.csv",
                        rows: int = 2500,
                        output_csv: str = "ldpc_predictions_fast.csv"):
    """
    Мини-прогон пайплайна на усечённых данных.
    Проверяет, что обучение и предсказание проходят без ошибок.
    """
    t0 = time.time()
    tmp_csv = "datasets/frames_errors_small.csv"

    df = pd.read_csv(input_csv, header=None)
    print(f"Оригинал: {df.shape[0]} строк, {df.shape[1]} колонок")
    if df.shape[1] != len(BASE_COLS):
        raise ValueError(
            f"В файле {df.shape[1]} колонок, в BASE_COLS {len(BASE_COLS)} имён. "
            "Списки должны совпадать."
        )

    df_small = df.iloc[:rows].copy()
    df_small.to_csv(tmp_csv, index=False, header=False)
    print(f"Срез сохранён -> {tmp_csv} ({df_small.shape})")

    pred_df = run_pipeline(input_csv=tmp_csv, output_csv=output_csv)

    print(f"\nМини-прогон завершён за {time.time()-t0:.1f} сек.")
    print(f"Форма результата: {pred_df.shape}")

    col = next((c for c in ["QBER_med_pred", "E_mu_Z_pred", "E_mu_Z"] if c in pred_df.columns), None)
    if col:
        print(f"\nСтатистика по {col}:")
        print(pred_df[col].describe())

    if all(c in pred_df.columns for c in ["R_pred", "s_pred", "p_pred"]):
        print(f"\nR_pred unique: {sorted(pred_df['R_pred'].unique())}")
        print(f"s_pred range: {int(pred_df['s_pred'].min())} - {int(pred_df['s_pred'].max())}")
        print(f"p_pred range: {int(pred_df['p_pred'].min())} - {int(pred_df['p_pred'].max())}")
        print(f"s+p==4800: {(pred_df['s_pred']+pred_df['p_pred']).eq(4800).all()}")

    return pred_df


pred_df = quick_test_pipeline()

In [ ]:
# ==== QUICK CHECK (минимум данных + реальные предикты) ====

# параметры целевого окна
start_bid, start_fr = 1489460492, 99
end_bid,   end_fr   = 1840064900, 101

# читаем ИСХОДНИК БЕЗ заголовка (важно!)
raw = pd.read_csv('datasets/frames_errors.csv', header=None)

# находим индексы начала/конца окна
i0 = raw.index[(raw.iloc[:,0]==start_bid) & (raw.iloc[:,1]==start_fr)][0]
i1 = raw.index[(raw.iloc[:,0]==end_bid)   & (raw.iloc[:,1]==end_fr)][0]
win_len = i1 - i0 + 1  # должно быть 2000

# TimeSeriesSplit у тебя: n_splits=5, test_size=400, gap=2 → нужно минимум 2003 строк
need_len = 5*400 + 2 + 1  # 2003
ctx = max(0, need_len - win_len)  # сколько добавить строк ДО окна

sl = raw.iloc[max(0, i0-ctx): i1+1].copy()     # контекст + окно
tmp_csv = 'datasets/_fast.csv'
sl.to_csv(tmp_csv, index=False, header=False)   # ВАЖНО: без header

# запускаем твой пайплайн на маленьком файле
pred_all = run_pipeline(input_csv=tmp_csv, output_csv='ldpc_predictions_fast.csv')

# берём РОВНО 2000 целевых строк из конца (отсекая контекст)
pred = pred_all.tail(win_len).copy()

# быстрая проверка: не константа ли шум
metric_col = 'QBER_med_pred' if 'QBER_med_pred' in pred.columns else (
             'E_mu_Z_pred' if 'E_mu_Z_pred' in pred.columns else 'E_mu_Z')
print(pred[metric_col].describe())

# проверяем ПРЕДСКАЗАННЫЕ параметры политики (именно *_pred)
if 'R_pred' in pred.columns:
    print("R_pred unique:", sorted(pred['R_pred'].unique()))
if 's_pred' in pred.columns:
    s = pred['s_pred']
    p = pred['p_pred'] if 'p_pred' in pred.columns else (4800 - s)
    print("s_pred range:", int(s.min()), "-", int(s.max()))
    print("p range:", int(p.min()), "-", int(p.max()))
    print("s_pred + p == 4800:", (s + p).eq(4800).all())





In [ ]:
# Прогон с демо-данными (если рядом нет data.csv). Создаст ldpc_predictions.csv
pred_df = run_pipeline(input_csv='datasets/frames_errors.csv', output_csv='ldpc_predictions.csv')
plot_R_vs_QBER(pred_df)
pred_df.head(10)

In [ ]:
pred_df

In [ ]:
sub4 = pred_df[["QBER_med_pred","R_pred","s_pred","p_pred"]].copy()
sub4 = sub4.rename(columns={"QBER_med_pred":"E_mu_Z_est",
                            "R_pred":"R",
                            "s_pred":"s",
                            "p_pred":"p"})
assert (sub4["s"] + sub4["p"] == 4800).all(), "s+p должно быть 4800"
sub4.to_csv("submission.csv", index=False, header=False)
print("Saved: submission.csv (4 cols, 2000 rows)")

In [ ]:
sub = pd.read_csv("submission.csv", header=None, names=["E_mu_Z","R","s","p"])
print(sub.describe())
print(sub.nunique())
print(sub.head(20))
print("Уникальные значения:", sub_out.nunique().to_dict())

In [ ]:
with open("submission.csv", "r") as f:
    for i in range(10):
        print(f.readline().strip())

In [ ]:
# === 1) Формируем шаблон для 2000 фреймов (надёжный способ) ===
raw = raw.sort_values(["block_id", "frame_idx"]).reset_index(drop=True)

# Определяем границы из условия соревнования
start_block, start_frame = 1489460492, 99
end_block, end_frame = 1840064900, 101

# Находим индекс начала и конца
start_idx = raw.index[(raw["block_id"] == start_block) & (raw["frame_idx"] == start_frame)][0]
end_idx   = raw.index[(raw["block_id"] == end_block) & (raw["frame_idx"] == end_frame)][0]

# Делаем срез
template = raw.loc[start_idx:end_idx].copy()

print(f"✅ Выбрано строк: {len(template)} (ожидали 2000)")
assert len(template) == 2000, f"Ожидали 2000 строк, получили {len(template)}"

In [ ]:
raw[["block_id","frame_idx"]].head(), raw[["block_id","frame_idx"]].tail()
print(raw["block_id"].dtype, raw["frame_idx"].dtype)
print(start_block, end_block)
print(mask.sum())

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Реальный QBER vs предсказанный
plt.figure(figsize=(12,4))
plt.plot(df['frame_idx'], df['E_mu_Z'], label='real QBER', lw=1)
plt.plot(pred_df['frame_idx'], pred_df['QBER_med_pred'], label='predicted QBER', lw=1)
plt.legend(); plt.grid(); plt.title('QBER real vs predicted'); plt.show()

# 2. Карта R(QBER)
plt.figure(figsize=(6,4))
plt.scatter(pred_df['QBER_hi_pred'], pred_df['R_pred'], s=8, alpha=0.7)
plt.xlabel('Predicted QBER (q_hi)'); plt.ylabel('Chosen code rate R')
plt.title('Policy response'); plt.grid(True); plt.show()

# 3. Важность признаков модели
if hasattr(forecaster.q50, "feature_importances_"):
    imp = pd.Series(forecaster.q50.feature_importances_, index=forecaster.feature_cols)
    imp.sort_values(ascending=False).head(15).plot(kind="barh", figsize=(8,5))
    plt.title("Top-15 feature importances (LGBM quantile)")
    plt.gca().invert_yaxis(); plt.grid(True); plt.show()

# 4. Корреляции физических параметров
plt.figure(figsize=(8,6))
sns.heatmap(df[['E_mu_Z','E_mu_X','temp_1','temp_2','biasVoltage_1','biasVoltage_2','opticalPower']].corr(), annot=True, fmt=".2f")
plt.title("Feature correlations"); plt.show()

## 10) Как использовать на своих данных

1. Положите ваш CSV в ту же папку, назовите его `data.csv` (или укажите путь в `run_pipeline`).
2. Убедитесь, что заголовки соответствуют формату (см. описание задачи). Минимально нужен хотя бы один из: `E_mu_Z` или `E_mu_Z_est`.
3. Запустите ячейку с `run_pipeline("data.csv", "ldpc_predictions.csv")`.
4. Заберите `ldpc_predictions.csv` — там будут колонки `R_pred, s_pred, p_pred` и предсказанные `QBER_*`.